In [1]:
import pandas as pd
import numpy as np
import re
from io import StringIO

In [2]:
file_path_1 = r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00106.csv'
file_path_2 = r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00106.csv'

with open(file_path_1, 'r', encoding='cp950', errors='ignore') as f1:
    content1 = f1.read()
with open(file_path_2, 'r', encoding='cp950', errors='ignore') as f2:
    content2 = f2.read()
    
df2024 = pd.read_csv(StringIO(content1), low_memory=False, dtype={'ORDERTIME': str, 'GATHERTIME': str, 'VERIFYTIME': str})
df2025 = pd.read_csv(StringIO(content2), low_memory=False, dtype={'ORDERTIME': str, 'GATHERTIME': str, 'VERIFYTIME': str})

len(df2024), len(df2025)

(1048575, 993767)

In [3]:
df2024['ORDERTIME'] = pd.to_datetime(df2024['ORDERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['GATHERTIME'] = pd.to_datetime(df2024['GATHERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['VERIFYTIME'] = pd.to_datetime(df2024['VERIFYTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ORDERTIME'] = pd.to_datetime(df2025['ORDERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['GATHERTIME'] = pd.to_datetime(df2025['GATHERTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['VERIFYTIME'] = pd.to_datetime(df2025['VERIFYTIME'], format='%Y%m%d%H%M%S', errors='coerce')

table6 = pd.concat([df2024, df2025], ignore_index=True)
print(len(table6))

2042342


In [4]:
table6 = table6.dropna(how='all')
table6 = table6[table6['ORDERTIME'].notna()]
table6 = table6.loc[:, ~table6.columns.str.contains('^Unnamed')]

In [5]:
table6

,DIVISIONNO,ACCOUNTNO,IPDACCOUNTNO,ORDERNO,LABITEMNO,ORDERNAME,ITEMNAME,RESULTVALUE,ORDERTIME,GATHERTIME,VERIFYTIME
1048575,001,I11400000003,NaN,L4000200,L40201,尿沉渣Urine Sediment,Microscopic RBC,0-2,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048576,001,I11400000003,NaN,L4000200,L40202,尿沉渣Urine Sediment,Microscopic WBC,20-29,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048577,001,I11400000003,NaN,L4000200,L40203,尿沉渣Urine Sediment,Ep.cell,0-5,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048578,001,I11400000003,NaN,L4000200,L40204,尿沉渣Urine Sediment,Cast(1),-,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
1048579,001,I11400000003,NaN,L4000200,L40205,尿沉渣Urine Sediment,Crystal (1),-,2025-01-01 00:31:00,2025-01-01 01:01:09,2025-01-01 01:03:01
...,...,...,...,...,...,...,...,...,...,...,...
2042336,001,I11400060739,NaN,L6001100,L61101,GOT,GOT,18,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042337,001,I11400060739,NaN,L6001700,L61701,Creatinine,Creatinine,0.54,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042338,001,I11400060739,NaN,L6001700,L61702,Creatinine,eGFR(MDRD計算公式，僅供參考),126.3,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05
2042339,001,I11400060739,NaN,L6001700,L61703,Creatinine,eGFR(CKD-EPI計算公式，僅供參考),120.8,2025-12-31 23:50:00,2025-12-31 23:05:08,2026-01-01 00:02:05


In [6]:
len(table6),len(table6['ACCOUNTNO'].unique())

(993766, 27112)

In [7]:
table6 = table6[['ACCOUNTNO','ITEMNAME', 'RESULTVALUE']]

In [8]:
# raw_text = """'
# '人工閱片', 'Neutrophil Seg.', 'Lymphocyte', 'Monocyte', 'Eosinophil',
#        'Basophil', 'Absolute Neutrophil count', 'Hb', 'Ht', 'RBC', 'RDW',
#        'WBC', 'MCV', 'MCH', 'MCHC', 'PLT', 'Na', 'K', 'Glucose', 'GPT',
# """

# # 簡單清理並轉成一維清單
# items = [i.strip().strip("'") for i in raw_text.replace('\n', ',').split(',') if i.strip()]

In [9]:
counts = table6['ITEMNAME'].value_counts()
threshold = len(table6['ACCOUNTNO'].unique()) * 0.2
top_items = counts[counts > threshold].index.tolist()
top_label = table6[table6['ITEMNAME'].isin(top_items)]['ITEMNAME'].unique()

In [10]:
top_label

array(['Microscopic RBC', 'Microscopic WBC', 'Ep.cell', 'Cast(1)',
       'Crystal (1)', 'Bacteria', 'Appearance', 'Glucose', 'Bilirubin',
       'Ketone Body', 'Sp.gr', 'OB', 'PH', 'Protein', 'Urobilinogen',
       'Nitrite', 'Leukocyte', '人工閱片', 'Neutrophil Seg.', 'Lymphocyte',
       'Monocyte', 'Eosinophil', 'Basophil', 'Absolute Neutrophil count',
       'Hb', 'Ht', 'RBC', 'RDW', 'WBC', 'MCV', 'MCH', 'MCHC', 'PLT', 'pH',
       'pCO2', 'pO2', 'HCO3 act', 'ctCO2', 'BE(b)', 'BE(ecf)', 'HCO3 std',
       'O2 SAT', 'Na', 'K', 'GPT', 'Creatinine', 'eGFR(MDRD計算公式，僅供參考)',
       'CRP', 'hs-Troponin T', 'T.Bilirubin', 'Lipase', 'PT', 'MNPT',
       'INR', 'APTT', 'MNaPTT', 'Influenza Virus A Screening Test',
       'Influenza Virus B screening test', '新型冠狀病毒抗原檢測SARS-CoV-2 Ag test',
       'eGFR(CKD-EPI計算公式，僅供參考)'], dtype=object)

In [11]:
wbc_df = table6[table6['ITEMNAME'] == 'WBC'].copy()

wbc_df['RESULTVALUE_wbc'] = (wbc_df['RESULTVALUE'].str.replace('<', '', regex=False).str.replace('>', '', regex=False))
wbc_df['RESULTVALUE_wbc'] = pd.to_numeric(wbc_df['RESULTVALUE_wbc'], errors='coerce')

wbc_pivot = wbc_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_wbc', aggfunc='max'
                              ).rename(columns={'RESULTVALUE_wbc': 'WBC'})

print(wbc_pivot.shape)

(23425, 1)


In [12]:
crp_df = table6[table6['ITEMNAME'] == 'CRP'].copy()
crp_df['with_below'] = crp_df['RESULTVALUE'].str.contains('<', na=False)

crp_df['clean_values'] = crp_df['RESULTVALUE'].str.replace('<', '', regex=False)
crp_df['clean_values'] = pd.to_numeric(crp_df['clean_values'], errors='coerce')

# 有<0.1這種就/2
crp_df['RESULTVALUE_crp'] = np.where(crp_df['with_below'], crp_df['clean_values'] / 2, crp_df['clean_values'])

crp_pivot = crp_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_crp', aggfunc='max').rename(columns={'RESULTVALUE_crp': 'CRP'})
print(crp_pivot.shape)

(13342, 1)


In [13]:
lym_df = table6[table6['ITEMNAME'] == 'Lymphocyte'].copy()
lym_df['RESULTVALUE_lym'] = pd.to_numeric(lym_df['RESULTVALUE'], errors='coerce')

lym_pivot = lym_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_lym', aggfunc='max').rename(columns={'RESULTVALUE_lym': 'Lymphocyte'})
print(lym_pivot.shape)

(23298, 1)


In [14]:
neu_df = table6[table6['ITEMNAME'] == 'Absolute Neutrophil count'].copy()
neu_df['RESULTVALUE_neu'] = pd.to_numeric(neu_df['RESULTVALUE'], errors='coerce')

neu_pivot = neu_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_neu', aggfunc='max').rename(columns={'RESULTVALUE_neu': 'Absolute Neutrophil count'})
print(neu_pivot.shape)

(23253, 1)


In [15]:
neu_seg_df = table6[table6['ITEMNAME'] == 'Neutrophil Seg.'].copy()
neu_seg_df['RESULTVALUE_neu_seg'] = pd.to_numeric(neu_seg_df['RESULTVALUE'], errors='coerce')

neu_seg_pivot = neu_seg_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_neu_seg', aggfunc='max').rename(columns={'RESULTVALUE_neu_seg': 'Neutrophil Seg.'})
print(neu_seg_pivot.shape)

(23276, 1)


In [16]:
cre_df = table6[table6['ITEMNAME'] == 'Creatinine'].copy()
cre_df['with_below'] = cre_df['RESULTVALUE'].str.contains('<', na=False)
cre_df['clean_val'] = cre_df['RESULTVALUE'].str.replace('>', '', regex=False).str.replace('<', '', regex=False)
cre_df['clean_val'] = pd.to_numeric(cre_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
cre_df['RESULTVALUE_cre'] = np.where(cre_df['with_below'], cre_df['clean_val'] / 2, cre_df['clean_val'])

cre_pivot = cre_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_cre', aggfunc='max').rename(columns={'RESULTVALUE_cre': 'Creatinine'})
print(cre_pivot.shape)

(23064, 1)


In [17]:
egfr_md_df = table6[table6['ITEMNAME'] == 'eGFR(MDRD計算公式，僅供參考)'].copy()
egfr_md_df['RESULTVALUE_egfr_md'] = pd.to_numeric(egfr_md_df['RESULTVALUE'], errors='coerce')

egfr_md_pivot = egfr_md_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_egfr_md', aggfunc='max').rename(columns={'RESULTVALUE_egfr_md': 'eGFR-MDRD'})
print(egfr_md_pivot.shape)

(21710, 1)


In [18]:
#2025才有
egfr_ckd_df = table6[table6['ITEMNAME'] == 'eGFR(CKD-EPI計算公式，僅供參考)'].copy()
egfr_ckd_df['RESULTVALUE_egfr_ckd'] = pd.to_numeric(egfr_ckd_df['RESULTVALUE'], errors='coerce')

egfr_ckd_pivot = egfr_ckd_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_egfr_ckd', aggfunc='max').rename(columns={'RESULTVALUE_egfr_ckd': 'eGFR-CKD-EPI'})
print(egfr_ckd_pivot.shape)

(8083, 1)


In [19]:
gpt_df = table6[table6['ITEMNAME'] == 'GPT'].copy()
gpt_df['with_below'] = gpt_df['RESULTVALUE'].str.contains('<', na=False)
gpt_df['clean_val'] = gpt_df['RESULTVALUE'].str.replace('>', '', regex=False).str.replace('<', '', regex=False)
gpt_df['clean_val'] = pd.to_numeric(gpt_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
gpt_df['RESULTVALUE_gpt'] = np.where(gpt_df['with_below'], gpt_df['clean_val'] / 2, gpt_df['clean_val'])

gpt_pivot = gpt_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_gpt', aggfunc='max').rename(columns={'RESULTVALUE_gpt': 'GPT'})
print(gpt_pivot.shape)

(22168, 1)


In [20]:
# df = table6[table6['ITEMNAME'] == 'Bilirubin'].copy()
# df[df['ITEMNAME'] == 'Bilirubin']['RESULTVALUE'].describe()

In [21]:
# df = table6[table6['ITEMNAME'] == 'T.Bilirubin'].copy()
# df[df['ITEMNAME'] == 'T.Bilirubin']['RESULTVALUE'].describe()

In [22]:
bil_df = table6[table6['ITEMNAME'] == 'T.Bilirubin'].copy()
bil_df['with_below'] = bil_df['RESULTVALUE'].str.contains('<', na=False)

bil_df['clean_values'] = bil_df['RESULTVALUE'].str.replace('<', '', regex=False)
bil_df['clean_values'] = pd.to_numeric(bil_df['clean_values'], errors='coerce')

# 有<0.1這種就/2
bil_df['RESULTVALUE_bil'] = np.where(bil_df['with_below'], bil_df['clean_values'] / 2, bil_df['clean_values'])
bil_pivot = bil_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_bil', aggfunc='max').rename(columns={'RESULTVALUE_bil': 'T.Bilirubin'})
print(bil_pivot.shape)

(5799, 1)


In [23]:
pt_df = table6[table6['ITEMNAME'] == 'PT'].copy()
pt_df['RESULTVALUE_pt'] = pt_df['RESULTVALUE'].str.replace('>', '', regex=False)
pt_df['RESULTVALUE_pt'] = pd.to_numeric(pt_df['RESULTVALUE_pt'], errors='coerce')

pt_pivot = pt_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_pt', aggfunc='max').rename(columns={'RESULTVALUE_pt': 'PT'})
print(pt_pivot.shape)

(4824, 1)


In [24]:
inr_df = table6[table6['ITEMNAME'] == 'INR'].copy()
inr_df['RESULTVALUE_inr'] = inr_df['RESULTVALUE'].str.replace('>', '', regex=False)
inr_df['RESULTVALUE_inr'] = pd.to_numeric(inr_df['RESULTVALUE_inr'], errors='coerce')

inr_pivot = inr_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_inr', aggfunc='max').rename(columns={'RESULTVALUE_inr': 'INR'})
print(inr_pivot.shape)

(4824, 1)


In [25]:
aptt_df = table6[table6['ITEMNAME'] == 'APTT'].copy()
aptt_df['with_below'] = aptt_df['RESULTVALUE'].str.contains('<', na=False)
aptt_df['clean_val'] = aptt_df['RESULTVALUE'].str.replace('>', '', regex=False).str.replace('<', '', regex=False)
aptt_df['clean_val'] = pd.to_numeric(aptt_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
aptt_df['RESULTVALUE_aptt'] = np.where(aptt_df['with_below'], aptt_df['clean_val'] / 2, aptt_df['clean_val'])

aptt_pivot = aptt_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_aptt', aggfunc='max').rename(columns={'RESULTVALUE_aptt': 'APTT'})
print(aptt_pivot.shape)

(4820, 1)


In [26]:
hst_df = table6[table6['ITEMNAME'] == 'hs-Troponin T'].copy()
hst_df['with_below'] = hst_df['RESULTVALUE'].str.contains('<', na=False)
hst_df['clean_val'] = hst_df['RESULTVALUE'].str.replace('>', '', regex=False).str.replace('<', '', regex=False)
hst_df['clean_val'] = pd.to_numeric(hst_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
hst_df['RESULTVALUE_hst'] = np.where(hst_df['with_below'], hst_df['clean_val'] / 2, hst_df['clean_val'])

hst_pivot = hst_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_hst', aggfunc='max').rename(columns={'RESULTVALUE_hst': 'HST'})
print(hst_pivot.shape)

(10220, 1)


In [27]:
ph_df = table6[table6['ITEMNAME'] == 'PH'].copy()
ph_df['with_below'] = ph_df['RESULTVALUE'].str.contains('<=', na=False)
ph_df['clean_val'] = ph_df['RESULTVALUE'].str.replace('>=', '', regex=False).str.replace('<=', '', regex=False)
ph_df['clean_val'] = pd.to_numeric(ph_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
ph_df['RESULTVALUE_ph'] = np.where(ph_df['with_below'], ph_df['clean_val'] / 2, ph_df['clean_val'])

ph_pivot = ph_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_ph', aggfunc='max').rename(columns={'RESULTVALUE_ph': 'PH'})
print(ph_pivot.shape)

(8018, 1)


In [28]:
pco2_df = table6[table6['ITEMNAME'] == 'pCO2'].copy()
pco2_df['with_below'] = pco2_df['RESULTVALUE'].str.contains('<', na=False)

pco2_df['clean_values'] = pco2_df['RESULTVALUE'].str.replace('<', '', regex=False)
pco2_df['clean_values'] = pd.to_numeric(pco2_df['clean_values'], errors='coerce')

# 有<0.1這種就/2
pco2_df['RESULTVALUE_pco2'] = np.where(pco2_df['with_below'], pco2_df['clean_values'] / 2, pco2_df['clean_values'])
pco2_pivot = pco2_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_pco2', aggfunc='max').rename(columns={'RESULTVALUE_pco2': 'PCO2'})
print(pco2_pivot.shape)

(3345, 1)


In [29]:
# df = table6[table6['ITEMNAME'] == 'HCO3 act'].copy()
# df[df['ITEMNAME'] == 'HCO3 act']['RESULTVALUE'].describe()

In [30]:
# df = table6[table6['ITEMNAME'] == 'HCO3 std'].copy()
# df[df['ITEMNAME'] == 'HCO3 std']['RESULTVALUE'].describe()

In [31]:
hco3_df = table6[table6['ITEMNAME'] == 'HCO3 act'].copy()
hco3_df['RESULTVALUE_hco3'] = pd.to_numeric(hco3_df['RESULTVALUE'], errors='coerce')

hco3_pivot = hco3_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_hco3', aggfunc='max').rename(columns={'RESULTVALUE_hco3': 'HCO3'})
print(hco3_pivot.shape)

(3241, 1)


In [32]:
be_ecf_df = table6[table6['ITEMNAME'] == 'BE(ecf)'].copy()
be_ecf_df['RESULTVALUE_be_ecf'] = pd.to_numeric(be_ecf_df['RESULTVALUE'], errors='coerce')

be_ecf_pivot = be_ecf_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_be_ecf', aggfunc='max').rename(columns={'RESULTVALUE_be_ecf': 'BE(ecf)'})
print(be_ecf_pivot.shape)

(3241, 1)


In [33]:
o2_sat_df = table6[table6['ITEMNAME'] == 'O2 SAT'].copy()
o2_sat_df['with_below'] = o2_sat_df['RESULTVALUE'].str.contains('<', na=False)

o2_sat_df['clean_values'] = o2_sat_df['RESULTVALUE'].str.replace('<', '', regex=False)
o2_sat_df['clean_values'] = pd.to_numeric(o2_sat_df['clean_values'], errors='coerce')

# 有<0.1這種就/2
o2_sat_df['RESULTVALUE_o2_sat'] = np.where(o2_sat_df['with_below'], o2_sat_df['clean_values'] / 2, o2_sat_df['clean_values'])
o2_sat_pivot = o2_sat_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_o2_sat', aggfunc='max').rename(columns={'RESULTVALUE_o2_sat': 'O2 SAT'})
print(o2_sat_pivot.shape)

(3231, 1)


In [34]:
na_df = table6[table6['ITEMNAME'] == 'Na'].copy()
na_df['with_below'] = na_df['RESULTVALUE'].str.contains('<=', na=False)
na_df['clean_val'] = na_df['RESULTVALUE'].str.replace('>=', '', regex=False).str.replace('<=', '', regex=False)
na_df['clean_val'] = pd.to_numeric(na_df['clean_val'], errors='coerce')

# 只有當 with_below 為 True 時才除以 2，否則保持原值
na_df['RESULTVALUE_na'] = np.where(na_df['with_below'], na_df['clean_val'] / 2, na_df['clean_val'])

na_pivot = na_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_na', aggfunc='max').rename(columns={'RESULTVALUE_na': 'Na'})
print(na_pivot.shape)

(23127, 1)


In [35]:
k_df = table6[table6['ITEMNAME'] == 'K'].copy()
k_df['RESULTVALUE_k'] = k_df['RESULTVALUE'].str.replace('>', '', regex=False)
k_df['RESULTVALUE_k'] = pd.to_numeric(k_df['RESULTVALUE_k'], errors='coerce')

k_pivot = k_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_k', aggfunc='max').rename(columns={'RESULTVALUE_k': 'K'})
print(k_pivot.shape)

(23122, 1)


In [36]:
# micro_rbc_df[micro_rbc_df['ACCOUNTNO']=='I11400000034']

In [37]:
micro_rbc_df = table6[table6['ITEMNAME'] == 'Microscopic RBC'].copy()

rbc_mapping = {
    '0-2': 1,
    '3-5': 2,      
    '6-9': 3,       
    '10-19': 4, 
    '20-29': 5,
    '30-49': 6,
    '50-99': 7,
    '>=100': 8
}

def map_rbc(val):
    if pd.isna(val): 
        return None  
    
    val_clean = str(val).strip()
    
    if val_clean in rbc_mapping:
        return rbc_mapping[val_clean]
    # 如果是其他字串，回傳 -1
    return -1

micro_rbc_df['Microscopic RBC level'] = micro_rbc_df['RESULTVALUE'].apply(map_rbc)
micro_rbc_df = micro_rbc_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(micro_rbc_df['Microscopic RBC level'].value_counts(dropna=False).sort_index())
print(micro_rbc_df.shape)

Microscopic RBC level
1    4807
2    1045
3     501
4     389
5     226
6     208
7     282
8     709
Name: count, dtype: int64
(8167, 2)


In [38]:
# 取最大的ACCOUNTNO，去重
micro_rbc_df = micro_rbc_df.groupby('ACCOUNTNO', as_index=False)['Microscopic RBC level'].max()

In [39]:
len(micro_rbc_df), len(micro_rbc_df['ACCOUNTNO'])

(8016, 8016)

In [40]:
micro_wbc_df = table6[table6['ITEMNAME'] == 'Microscopic WBC'].copy()

wbc_mapping = {
    '0-5': 1,
    '6-9': 2,             
    '10-19': 3, 
    '20-29': 4,
    '30-49': 5,
    '50-99': 6,
    '>=100': 7
}

def map_wbc(val):
    if pd.isna(val): 
        return None  
    
    val_clean = str(val).strip()
    
    if val_clean in wbc_mapping:
        return wbc_mapping[val_clean]
    return -1

micro_wbc_df['Microscopic WBC level'] = micro_wbc_df['RESULTVALUE'].apply(map_wbc)
micro_wbc_df = micro_wbc_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(micro_wbc_df['Microscopic WBC level'].value_counts(dropna=False).sort_index())
print(micro_wbc_df.shape)

Microscopic WBC level
1    4664
2    1172
3     564
4     368
5     339
6     376
7     684
Name: count, dtype: int64
(8167, 2)


In [41]:
# 取最大的ACCOUNTNO，去重
micro_wbc_df = micro_wbc_df.groupby('ACCOUNTNO', as_index=False)['Microscopic WBC level'].max()

In [42]:
bac_df = table6[table6['ITEMNAME'] == 'Bacteria'].copy()

bac_mapping = {
    '-': 1,
    '+/-': 2,             
    '1+': 3, 
    '2+': 4,
    '3+': 5,
    '4+': 6,
}

def map_bac(val):
    if pd.isna(val): 
        return None  
    
    val_clean = str(val).strip()
    
    if val_clean in bac_mapping:
        return bac_mapping[val_clean]
    return -1

bac_df['Bacteria level'] = bac_df['RESULTVALUE'].apply(map_bac)
bac_df = bac_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(bac_df['Bacteria level'].value_counts(dropna=False).sort_index())
print(bac_df.shape)

Bacteria level
1    2617
2    1579
3    1400
4    1267
5     651
6     653
Name: count, dtype: int64
(8167, 2)


In [43]:
# 取最大的ACCOUNTNO，去重
bac_df = bac_df.groupby('ACCOUNTNO', as_index=False)['Bacteria level'].max()

In [44]:
nit_df = table6[table6['ITEMNAME'] == 'Nitrite'].copy()

nit_mapping = {'-': 0, '+': 1}

def map_nit(val):
    if pd.isna(val):
        return None
    val_clean = str(val).strip()
    
    if val_clean in nit_mapping:
        return nit_mapping[val_clean]
    return -1

nit_df['Nitrite level'] = nit_df['RESULTVALUE'].apply(map_nit)
nit_df = nit_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(nit_df['Nitrite level'].value_counts(dropna=False).sort_index())
print(nit_df.shape)

Nitrite level
0    7186
1     983
Name: count, dtype: int64
(8169, 2)


In [45]:
# 取最大的ACCOUNTNO，去重
nit_df = nit_df.groupby('ACCOUNTNO', as_index=False)['Nitrite level'].max()

In [46]:
len(nit_df),len(nit_df['ACCOUNTNO'].unique())

(8018, 8018)

In [47]:
leu_df = table6[table6['ITEMNAME'] == 'Leukocyte'].copy()

leu_mapping = {'-': 0, '+/-': 1, '1+': 3, '2+': 4, '3+': 5}

def map_leu(val):
    if pd.isna(val):
        return None
    val_clean = str(val).strip()
    
    if val_clean in leu_mapping:
        return leu_mapping[val_clean]
    return -1

leu_df['Leukocyte level'] = leu_df['RESULTVALUE'].apply(map_leu)
leu_df = leu_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(leu_df['Leukocyte level'].value_counts(dropna=False).sort_index())
print(leu_df.shape)

Leukocyte level
0    3740
1    1373
3    1011
4     951
5    1094
Name: count, dtype: int64
(8169, 2)


In [48]:
# 取最大的ACCOUNTNO，去重
leu_df = leu_df.groupby('ACCOUNTNO', as_index=False)['Leukocyte level'].max()

In [49]:
len(leu_df),len(leu_df['ACCOUNTNO'].unique())

(8018, 8018)

In [50]:
hb_df = table6[table6['ITEMNAME'] == 'Hb'].copy()
hb_df['RESULTVALUE_hb'] = pd.to_numeric(hb_df['RESULTVALUE'], errors='coerce')

hb_pivot = hb_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_hb', aggfunc='max').rename(columns={'RESULTVALUE_hb': 'Hb'})
print(hb_pivot.shape)

(23428, 1)


In [51]:
ht_df = table6[table6['ITEMNAME'] == 'Ht'].copy()
ht_df['RESULTVALUE_ht'] = pd.to_numeric(ht_df['RESULTVALUE'], errors='coerce')

ht_pivot = ht_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_ht', aggfunc='max').rename(columns={'RESULTVALUE_ht': 'Ht'})
print(ht_pivot.shape)

(23422, 1)


In [52]:
plt_df = table6[table6['ITEMNAME'] == 'PLT'].copy()
plt_df['RESULTVALUE_plt'] = pd.to_numeric(plt_df['RESULTVALUE'], errors='coerce')

plt_pivot = plt_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_plt', aggfunc='max').rename(columns={'RESULTVALUE_plt': 'PLT'})
print(plt_pivot.shape)

(23384, 1)


In [53]:
ivA_df = table6[table6['ITEMNAME'] == 'Influenza Virus A Screening Test'].copy()

ivA_mapping = {'Negative': 0, 'Positive': 1}

def map_ivA(val):
    if pd.isna(val):
        return None
    val_clean = str(val).strip()
    
    if val_clean in ivA_mapping:
        return ivA_mapping[val_clean]
    return -1

ivA_df['Influenza Virus A level'] = ivA_df['RESULTVALUE'].apply(map_ivA)
ivA_df = ivA_df.drop(columns=['ITEMNAME', 'RESULTVALUE'])
print(ivA_df['Influenza Virus A level'].value_counts(dropna=False).sort_index())
print(ivA_df.shape)

Influenza Virus A level
0    5826
1     869
Name: count, dtype: int64
(6695, 2)


In [54]:
# 取最大的ACCOUNTNO，去重
ivA_df = ivA_df.groupby('ACCOUNTNO', as_index=False)['Influenza Virus A level'].max()

In [55]:
# ivB_df = table6[table6['ITEMNAME'] == 'Influenza Virus B Screening Test'].copy()

# ivB_mapping = {'Negative': 0, 'Positive': 1}

# def map_ivB(val):
#     if pd.isna(val):
#         return None
#     val_clean = str(val).strip()
    
#     if val_clean in ivB_mapping:
#         return ivB_mapping[val_clean]
#     return -1

# ivB_df['Influenza Virus B level'] = ivB_df['RESULTVALUE'].apply(map_ivB)
# print(ivB_df['Influenza Virus B level'].value_counts(dropna=False).sort_index())

In [56]:
# sars_df = table6[table6['ITEMNAME'] == 'SARS-CoV-2 Ag'].copy()

# sars_mapping = {'Negative': 0, 'Positive': 1}

# def map_sars(val):
#     if pd.isna(val):
#         return None
#     val_clean = str(val).strip()
    
#     if val_clean in sars_mapping:
#         return sars_mapping[val_clean]
#     return -1

# sars_df['SARS-CoV-2 Ag level'] = sars_df['RESULTVALUE'].apply(map_sars)
# print(sars_df['SARS-CoV-2 Ag level'].value_counts(dropna=False).sort_index())

In [57]:
# trop_df = table6[table6['ITEMNAME'] == 'troponin'].copy()
# trop_df['with_below'] = trop_df['RESULTVALUE'].str.contains('<=', na=False)
# trop_df['clean_val'] = trop_df['RESULTVALUE'].str.replace('>=', '', regex=False).str.replace('<=', '', regex=False)
# trop_df['clean_val'] = pd.to_numeric(trop_df['clean_val'], errors='coerce')

# # 只有當 with_below 為 True 時才除以 2，否則保持原值
# trop_df['RESULTVALUE_trop'] = np.where(trop_df['with_below'], trop_df['clean_val'] / 2, trop_df['clean_val'])

# trop_pivot = trop_df.pivot_table(index='ACCOUNTNO', values='RESULTVALUE_trop', aggfunc='max').rename(columns={'RESULTVALUE_trop': 'troponin'})

In [58]:
merge_df = hb_pivot.merge(wbc_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(ht_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(plt_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(lym_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(neu_seg_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(neu_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(na_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(k_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(cre_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(gpt_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(egfr_md_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(crp_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(hst_pivot, on='ACCOUNTNO', how='left')

merge_df = merge_df.merge(leu_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(nit_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(bac_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(micro_rbc_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(micro_wbc_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(egfr_ckd_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(ph_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(ivA_df, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(bil_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(pt_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(inr_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(aptt_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(pco2_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(hco3_pivot, on='ACCOUNTNO', how='left')
merge_df = merge_df.merge(be_ecf_pivot, on='ACCOUNTNO', how='left')
table6_clear = merge_df.merge(o2_sat_pivot, on='ACCOUNTNO', how='left')
print(table6_clear.shape)

(23428, 31)


In [59]:
table6_clear

,ACCOUNTNO,Hb,WBC,Ht,PLT,Lymphocyte,Neutrophil Seg.,Absolute Neutrophil count,Na,K,...,PH,Influenza Virus A level,T.Bilirubin,PT,INR,APTT,PCO2,HCO3,BE(ecf),O2 SAT
0,I11400000004,8.5,6.11,24.8,101.0,15.9,78.0,4770.0,143.0,4.2,...,5.5,NaN,NaN,NaN,NaN,NaN,43.3,18.9,-8.2,98.8
1,I11400000005,13.3,5.66,38.3,244.0,32.3,56.7,3210.0,135.0,3.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I11400000008,12.4,14.90,37.0,244.0,4.5,91.3,13610.0,140.0,3.6,...,5.5,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I11400000016,14.8,10.20,42.7,307.0,42.8,51.8,5280.0,141.0,3.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I11400000017,11.6,11.30,34.5,198.0,17.7,77.1,8710.0,137.0,3.9,...,5.5,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23423,I11400060720,18.3,11.48,53.6,174.0,13.2,78.8,9050.0,141.0,4.3,...,NaN,NaN,NaN,NaN,NaN,NaN,87.6,33.4,8.8,97.5
23424,I11400060722,14.1,10.70,41.8,377.0,34.2,47.9,5125.0,138.0,4.0,...,NaN,NaN,0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23425,I11400060731,9.8,12.42,30.1,224.0,7.6,91.7,11370.0,144.0,3.8,...,2.5,NaN,NaN,12.4,1.08,25.8,50.3,26.0,1.1,99.6
23426,I11400060737,12.8,5.38,37.4,226.0,34.2,55.0,2960.0,136.0,3.5,...,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
len(table6_clear), len(table6_clear['ACCOUNTNO'])

(23428, 23428)

In [63]:
table6_clear.to_csv('table6_clear.csv', index=False)